# Session 3: MilkLab RAG Retrieval Evaluation (Precision@3 & Recall@3)

สมุดบันทึกนี้ประเมินประสิทธิภาพของระบบ Retrieval ของ MilkLab° RAG Chatbot ด้วย 10 ชุดคำถามภาษาไทย พร้อมวัดค่า Precision@3 และ Recall@3 และแสดงแจกแจง Similarity Score

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from app import load_index

# โหลดโมเดลและ FAISS Index จาก app.py
model, index, chunks = load_index()
print(f"โหลด Chunks ทั้งหมด {len(chunks)} รายการสำเร็จ")

In [ ]:
# 1. เตรียม Ground Truth (10 คำถามภาษาไทย)
EVAL_DATASET = [
    {
        "query": "ร้านเปิดกี่โมงและปิดกี่โมง",
        "ground_truth": ["[เกี่ยวกับร้าน] MilkLab° เป็นร้านนมสดกลางคืน เปิดทุกวันยกเว้นจันทร์ เวลา 20:00 ถึง 01:00 น."]
    },
    {
        "query": "นมหมีฮอกไกโดราคาเท่าไหร่และขนาดเท่าไหร่",
        "ground_truth": ["[เมนูหลัก] นมหมีฮอกไกโด: 65 บาท (นมสดฮอกไกโด + วิปครีม) ขนาด 350 ml"]
    },
    {
        "query": "นมโกโก้บราวนี่ใส่อะไรบ้างราคาเท่าไหร่",
        "ground_truth": ["[เมนูหลัก] นมโกโก้บราวนี่: 70 บาท (นมสด + ผงโกโก้พรีเมียม + ก้อนบราวนี่) ขนาด 400 ml"]
    },
    {
        "query": "มีส่วนผสมของถั่วไหม ลูกค้าแพ้ถั่วกินได้ไหม",
        "ground_truth": ["[Allergen] ลูกค้าแพ้ถั่ว: เมนูทั้งหมดปลอดถั่ว"]
    },
    {
        "query": "มีเมนูไหนใส่วิปครีมบ้างและมีสารแพ้อะไร",
        "ground_truth": [
            "[เมนูหลัก] นมหมีฮอกไกโด: 65 บาท (นมสดฮอกไกโด + วิปครีม) ขนาด 350 ml",
            "[Allergen] วิปครีมมี dairy"
        ]
    },
    {
        "query": "ค่าส่งเท่าไหร่และส่งไกลแค่ไหน",
        "ground_truth": ["[FAQ] **ส่งได้ไกลแค่ไหน**: รัศมี 5 กม. ค่าส่ง 30 บาท"]
    },
    {
        "query": "คนกินมังสวิรัติหรือวีแกนทานนมร้านนี้ได้ไหม",
        "ground_truth": ["[FAQ] **กิน vegan ได้ไหม**: ไม่มีเมนู vegan ทุกเมนูใส่นมวัว"]
    },
    {
        "query": "สั่งจองเมนูล่วงหน้าทางไหนสั่งได้ถึงกี่โมง",
        "ground_truth": ["[FAQ] **จองล่วงหน้าได้ไหม**: ได้ ผ่าน LINE OA สั่งก่อน 19:00"]
    },
    {
        "query": "การสั่งซื้อมีกำหนดออเดอร์ขั้นต่ำหรือไม่",
        "ground_truth": ["[FAQ] **ออเดอร์ขั้นต่ำ**: ไม่มี"]
    },
    {
        "query": "นมเสาวรสส่วนผสมมีอะไรและราคาเท่าไหร่",
        "ground_truth": ["[เมนูหลัก] นมเสาวรส: 60 บาท (นมสด + น้ำเสาวรสสด) ขนาด 350 ml"]
    }
]

In [ ]:
# 2. รัน Retrieval Top-k (k=3) และคำนวณ Precision@3 และ Recall@3
k = 3
precisions = []
recalls = []
top1_scores = []

for i, item in enumerate(EVAL_DATASET, 1):
    query = item["query"]
    gt = item["ground_truth"]
    
    q_emb = model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    scores, indices = index.search(q_emb, k)
    
    retrieved_chunks = [chunks[idx] for idx in indices[0] if 0 <= idx < len(chunks)]
    top1_score = float(scores[0][0])
    top1_scores.append(top1_score)
    
    hits = sum(1 for c in retrieved_chunks if c in gt)
    precision = hits / k
    recall = hits / len(gt) if len(gt) > 0 else 0.0
    
    precisions.append(precision)
    recalls.append(recall)
    
    print(f"[Case {i}] Query: '{query}'")
    print(f"   Hits: {hits}/{len(gt)} | Precision@{k}: {precision:.4f} | Recall@{k}: {recall:.4f} | Top-1 Score: {top1_score:.4f}")

mean_precision = np.mean(precisions)
mean_recall = np.mean(recalls)

print("\n" + "="*60)
print(f"Overall Mean Precision@{k}: {mean_precision:.4f} ({mean_precision*100:.2f}%)")
print(f"Overall Mean Recall@{k}:    {mean_recall:.4f} ({mean_recall*100:.2f}%)")
print(f"Overall Mean Top-1 Score: {np.mean(top1_scores):.4f}")
print("="*60)

In [ ]:
# 3. พล็อต Histogram แจกแจงความถี่ของ Top-1 Similarity Score
plt.figure(figsize=(8, 5))
plt.hist(top1_scores, bins=5, color='teal', edgecolor='black', alpha=0.7)
plt.title('Distribution of Top-1 Similarity Scores (MilkLab RAG)')
plt.xlabel('Cosine Similarity Score')
plt.ylabel('Frequency')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()